In [1]:
%%capture
import warnings
warnings.filterwarnings("ignore")

from IPython.display import HTML, Markdown, display

import altair as alt
import calitp_portfolio.magics
import geopandas as gpd
import pandas as pd
from IPython.display import HTML, Image, Markdown, display, display_html
from great_tables import GT
import gt_extras as gte
import branca.colormap as cm

# from shared_utils import webmap_utils
from update_vars import DIGEST_DICT as GTFS_DATA_DICT, analysis_month

from omegaconf import OmegaConf
readable_dict = OmegaConf.load("readable.yml")

import google.auth
credentials, project = google.auth.default()
import _operator_report_utils as utils
from datetime import datetime
import gcsfs

In [2]:
import itables
itables.init_notebook_mode()
from itables import show, options

options.allow_html = True


In [3]:
from dotenv import dotenv_values

In [4]:
from calitp_data_analysis.gcs_pandas import GCSPandas
from functools import cache

In [5]:
@cache
def gcs_pandas():
    return GCSPandas()

In [6]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

In [7]:
def formatted(number):
    return "{:,}".format(number)

In [8]:
analysis_name = "Alameda-Contra Costa Transit District"

In [9]:
# %%capture_parameters
# analysis_name

In [10]:
# Set drop down menu to be on the upper right for the charts
display(
    HTML(
        """
<style>
form.vega-bindings {
  position: absolute;
  right: 0px;
  top: 0px;
}
</style>
"""
    )
)

In [11]:
GCS_PATH = f"{GTFS_DATA_DICT.dir}processed/"

In [12]:
analysis_name_edited = analysis_name.replace(" ","_").lower()
operator_webmap_file = f"{analysis_name_edited}_routes"

In [13]:
dt = datetime.strptime(analysis_month, "%Y-%m-%d")
analysis_month_for_filtering = dt.strftime("%m/%Y")

In [14]:
# Dataset 1
fct_monthly_routes_url = f"{GCS_PATH}{GTFS_DATA_DICT.route_map}.parquet"
fct_monthly_route_df = gpd.read_parquet(fct_monthly_routes_url,
                                        filters=[[("Analysis Name", "==", analysis_name)]],
                                        storage_options={"token": credentials.token}).reset_index()
fct_monthly_route_df['Number'] = fct_monthly_route_df.index

In [15]:
# Dataset 2
schedule_rt_route_direction_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.schedule_rt_route_direction}.parquet"
schedule_rt_route_direction_summary_df = gcs_pandas().read_parquet(schedule_rt_route_direction_summary_url,
                         filters=[[("Analysis Name", "==", analysis_name)]])
one_month_schedule_rt_route_direction_summary_df = (schedule_rt_route_direction_summary_df.loc[schedule_rt_route_direction_summary_df["Day Type"] == "Weekday"]
    .sort_values(by = ["Date", "Route"], ascending = [False, True])
    .drop_duplicates(subset = ["Route"])
                                                   )

In [16]:
# Dataset 3 
operator_hourly_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.hourly_day_type_summary}.parquet"
operator_hourly_summary_df = gcs_pandas().read_parquet(operator_hourly_summary_url,
                                             filters=[[("Analysis Name", "==", analysis_name),
                                                      ("Departure Hour", "<=", 24)]])

In [17]:
# Dataset 4 
ntd_url = f"{GCS_PATH}{GTFS_DATA_DICT.ntd_profile}.parquet"
ntd_profile_df =  gcs_pandas().read_parquet(ntd_url, filters=[[("analysis_name", "==", analysis_name)]])

In [18]:
# Dataset 5 
operator_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.operator_summary}.parquet"
operator_df = (gcs_pandas().read_parquet(
    operator_summary_url,
    filters=[
        ("Day Type", "==", "Weekday"),
        ("Analysis Name", "==", analysis_name),
        ("VP Name", "!=", "None")],
)).sort_values(by = ["Date","Daily TU Trips"], ascending = [False,False]).drop_duplicates(subset = [
    "Analysis Name","Date"
])
operator_df["Pct TU Trips"] = operator_df["Pct TU Trips"].round(2)

# {analysis_name}

## Operator Overview

In [19]:
try:
    service_area = formatted(ntd_profile_df.service_area_sq_miles.values[0])
    service_pop = formatted(ntd_profile_df.service_area_pop.values[0])
except:
    pass

In [20]:
try:
    display(
        Markdown(
            f"""{analysis_name} is headquartered in <b>{ntd_profile_df.county_name.values[0]}</b> 
            County in the Urbanized Area of <b>{ntd_profile_df.primary_uza_name.values[0]}</b>.<br>
            This operator provides <b>{service_area}</b> square miles of public transit service, 
            which has a service population of <b>{service_pop}</b>.<br>
            This organization is a {ntd_profile_df.reporter_type.values[0]}.<br>
            <b>Data Source</b>: 
            <a href="https://www.transit.dot.gov/ntd/data-product/2022-annual-database-agency-information">National Transit Database</a> 
            Annual Agency Information.
            """
        )
    )
except:
    pass

Alameda-Contra Costa Transit District is headquartered in <b>Alameda</b> 
            County in the Urbanized Area of <b>San Francisco--Oakland, CA</b>.<br>
            This operator provides <b>364</b> square miles of public transit service, 
            which has a service population of <b>1,586,454</b>.<br>
            This organization is a Full Reporter.<br>
            <b>Data Source</b>: 
            <a href="https://www.transit.dot.gov/ntd/data-product/2022-annual-database-agency-information">National Transit Database</a> 
            Annual Agency Information.
            

## Route Typologies

In [21]:
try:
    n_routes = one_month_schedule_rt_route_direction_summary_df["Route"].nunique()
    display(
        Markdown(
            f"""{analysis_name} runs <b>{n_routes}</b> unique routes. Below is the breakdown of the routes.<p>
            Route categories are determined by a combination of GTFS `route_type` 
            and `route_short_name` and `route_long_name`. 
            Please see the 
            <a href="https://github.com/cal-itp/gtfs-curator/blob/main/gtfs_digest/methodology.md">methodology docs</a>
            for more details on this approach.
        """
        )
    )
except:
    display(Markdown(f"""{analysis_name} doesn't have an operator profile."""))

Alameda-Contra Costa Transit District runs <b>125</b> unique routes. Below is the breakdown of the routes.<p>
            Route categories are determined by a combination of GTFS `route_type` 
            and `route_short_name` and `route_long_name`. 
            Please see the 
            <a href="https://github.com/cal-itp/gtfs-curator/blob/main/gtfs_digest/methodology.md">methodology docs</a>
            for more details on this approach.
        

In [22]:
try:
    display(utils.create_route_typology(one_month_schedule_rt_route_direction_summary_df))
except:
    display(Markdown(f"""{analysis_name} doesn't have information on route typologies."""))

alt.Chart(...)

## Route Length

In [23]:
try:
    display(utils.create_route_lengths(fct_monthly_route_df))
except:
    display(Markdown(f"""{analysis_name} doesn't have information on route lengths."""))

alt.Chart(...)

## Service Area

In [24]:
color_map = cm.linear.Spectral_11.scale(0, len(fct_monthly_route_df))

In [25]:
CARTO_TILE_URL = dotenv_values("../_env")["CARTO_TILE_URL"]

In [26]:
try:
    display(fct_monthly_route_df.drop_duplicates(subset = ["Route Name"])[["Route Name","Geometry", "Number"]].explore(
    "Route Name",
    cmap =  "Spectral",
    tiles = CARTO_TILE_URL,
    attr="Basemap",
    name = "Caltrans District Boundary",
    style_kwds = {"weight": 1, "fillOpacity": 0},
    highlight_kwds = {"fillOpacity": 0.2},
    tooltip=False,
    legend = False,
))
except:
    display(Markdown(f"""{analysis_name} doesn't have an route geographies."""))

## Service Hours 

In [27]:
try:
    display(utils.create_hourly_summary(operator_hourly_summary_df))
except: 
    display(Markdown(f"""{analysis_name} doesn't have service hours information."""))

alt.LayerChart(...)

## Trip Update Overview

In [28]:
try:
    display((
    GT(operator_df[["Date","TU Messages Per Minute", 'Pct TU Trips',]].sort_values("Date", ascending=False))
    .fmt_integer(
        columns=["TU Messages Per Minute", 'Pct TU Trips',], 
    ).tab_header(
        title="Trip Updates",
    ).cols_align(
        columns=["TU Messages Per Minute", 'Pct TU Trips',],
        align="center",
    ).pipe(
        gte.gt_color_box, 
        columns=["TU Messages Per Minute", 'Pct TU Trips',],
        palette="YlGnBu", 
    )
))
except:
    display(Markdown(f"""{analysis_name} doesn't have trip update information."""))

GT(_tbl_data=        Date  TU Messages Per Minute  Pct TU Trips
0 2026-11-01                    2.90         97.90
2 2026-10-01                    2.90         97.60
1 2026-09-01                    2.90         98.60, _body=<great_tables._gt_data.Body object at 0x7e536c79a6d0>, _boxhead=Boxhead([ColInfo(var='Date', type=<ColInfoTypeEnum.default: 1>, column_label='Date', column_align='right', column_width=None), ColInfo(var='TU Messages Per Minute', type=<ColInfoTypeEnum.default: 1>, column_label='TU Messages Per Minute', column_align='center', column_width=None), ColInfo(var='Pct TU Trips', type=<ColInfoTypeEnum.default: 1>, column_label='Pct TU Trips', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5357db8810>, _spanners=Spanners([]), _heading=Heading(title='Trip Updates', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e536c79b990>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e536c79a010>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e536c799f50>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e536c798690>, <great_tables._gt_data.FormatInfo object at 0x7e536c792e10>, <great_tables._gt_data.FormatInfo object at 0x7e536c793f10>, <great_tables._gt_data.FormatInfo object at 0x7e536c793610>, <great_tables._gt_data.FormatInfo object at 0x7e536c799290>, <great_tables._gt_data.FormatInfo object at 0x7e5356c8bc50>, <great_tables._gt_data.FormatInfo object at 0x7e5356c892d0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=Op

## Vehicle Positions Overview

In [29]:
try:
    display((
    GT(operator_df[["Date","VP Messages Per Minute",
                "Pct VP Trips"]].sort_values("Date", ascending=False))
    .fmt_integer(
        columns=["VP Messages Per Minute",
                "Pct VP Trips"], 
    ).tab_header(
        title="Vehicle Positions",
    ).cols_align(
        columns=["VP Messages Per Minute",
                "Pct VP Trips"],
        align="center",
    ).pipe(
        gte.gt_color_box, 
        columns=["VP Messages Per Minute",
                "Pct VP Trips"],
        palette="YlGnBu", 
    )
))
except:
    display(Markdown(f"""{analysis_name} doesn't have vehicle positions information."""))

GT(_tbl_data=        Date  VP Messages Per Minute  Pct VP Trips
0 2026-11-01                    2.90         89.50
2 2026-10-01                    2.90         87.70
1 2026-09-01                    2.90         90.40, _body=<great_tables._gt_data.Body object at 0x7e5356c89450>, _boxhead=Boxhead([ColInfo(var='Date', type=<ColInfoTypeEnum.default: 1>, column_label='Date', column_align='right', column_width=None), ColInfo(var='VP Messages Per Minute', type=<ColInfoTypeEnum.default: 1>, column_label='VP Messages Per Minute', column_align='center', column_width=None), ColInfo(var='Pct VP Trips', type=<ColInfoTypeEnum.default: 1>, column_label='Pct VP Trips', column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x7e5358286f90>, _spanners=Spanners([]), _heading=Heading(title='Vehicle Positions', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7e5356c8a550>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7e5356c8b290>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x7e5356c8abd0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x7e536c79bbd0>, <great_tables._gt_data.FormatInfo object at 0x7e536c7985d0>, <great_tables._gt_data.FormatInfo object at 0x7e536c799dd0>, <great_tables._gt_data.FormatInfo object at 0x7e536c799290>, <great_tables._gt_data.FormatInfo object at 0x7e536c799cd0>, <great_tables._gt_data.FormatInfo object at 0x7e5356c69210>, <great_tables._gt_data.FormatInfo object at 0x7e5356c68050>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_col

## Detailed Route Overview

In [30]:
schedule_rt_route_direction_summary_df2 = schedule_rt_route_direction_summary_df[["Date",
"Route","Direction", "Daily Trips All Day", "Average Scheduled Minutes", "Headway Peak", "Headway Offpeak"]]

In [31]:
schedule_rt_route_direction_summary_df2 = schedule_rt_route_direction_summary_df2.sort_values(by =["Date", "Route","Direction","Daily Trips All Day"], ascending = [False, False, False, False]).drop_duplicates(subset = ["Date","Route","Direction"])

In [32]:
            
try:
    show(schedule_rt_route_direction_summary_df2,
    layout={"top1": "searchPanes"},
    searchPanes={"layout": "columns-3", "cascadePanes": True, "columns": [1, 2]}
)
except:
    display(Markdown(f"""{analysis_name} doesn't have detailed route information."""))

Loading ITables v2.7.3 from the init_notebook_mode cell... (need help?)
